[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoeBosi/pietre-di-vita-demo/blob/main/notebooks/pietre_di_vita_verifica_it.ipynb)

# Frantoio Pietre di Vita: verifica una bottiglia con Python

Questo notebook ripete, passo dopo passo, la verifica che fa la pagina pubblica di una bottiglia: calcola l'impronta (hash SHA-256) di un documento, legge il lotto **25B001** dal contratto sulla rete di prova **Polygon Amoy** e controlla che l'impronta scritta sulla catena coincida con quella del documento scaricato.

**Cosa serve:** un account Google gratuito per eseguire le celle in Colab (menu *Runtime > Esegui tutto*). Tutte le letture sono pubbliche, quindi bastano il browser e l'account Google. Il frantoio, i dati e la rete sono di prova.

[English version](https://colab.research.google.com/github/JoeBosi/pietre-di-vita-demo/blob/main/notebooks/pietre_di_vita_verify_en.ipynb)

## 1. Installa le librerie

Una sola riga: `web3` parla con la blockchain, `requests` con i siti web.

In [1]:
!pip install -q web3 requests

## 2. Valori e funzioni di aiuto

L'indirizzo del contratto e quello del nodo pubblico sono quelli del sito. Le richieste web portano sempre un `User-Agent` esplicito, un tempo massimo e un messaggio di errore comprensibile.

In [2]:
import hashlib
import requests
from web3 import Web3

SITE = "https://tracciabilita-olio.blockchain.giuseppebosi.com"
RPC_URL = "https://polygon-amoy-bor-rpc.publicnode.com"      # free public Polygon Amoy node
CONTRACT = "0x158a5026eD83888a443f2bA921Faf07C2D04bE0E"     # PietreDiVita contract on Amoy
LOT = "25B001"

# Explicit headers: many public services answer 403 to requests that have no User-Agent.
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; educational-demo)", "Accept": "application/json"}


def fetch(url):
    """GET a URL with headers and a time limit; raise a readable error if something goes wrong."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
    except requests.RequestException as e:
        raise RuntimeError(f"Could not reach {url}: {e}")
    if r.status_code == 429:
        raise RuntimeError("Too many requests (HTTP 429): wait a minute and run the cell again.")
    if r.status_code != 200:
        raise RuntimeError(f"{url} answered HTTP {r.status_code}")
    return r


def sha256_hex(data: bytes) -> str:
    """SHA-256 fingerprint as 64 hexadecimal characters."""
    return hashlib.sha256(data).hexdigest()

## 3. L'impronta di un testo

Stessa prova del capitolo 3: due testi quasi uguali danno impronte completamente diverse. **Cosa dovresti vedere:** le due righe del capitolo 3 (`a65f4491...` e `801251e0...`) e `same? False`.

In [3]:
same_text = sha256_hex(b"acidita 0.2")
other_text = sha256_hex(b"acidita 0.3")
print(same_text)
print(other_text)
print("same?", same_text == other_text)

a65f4491f8a94984e9992a249393423c32d511f7ff15ee7bf83663c663ee54da
801251e0ce82939caa01ea32b7958cf57ecdac51422051fcff26c21d165b20c1
same? False


## 4. Leggi la bottiglia dall'API pubblica del sito

Una richiesta GET, la stessa che fa la pagina aperta dal QR. **Cosa dovresti vedere:** bottiglia 1 di 360 del lotto 25B001 e le quattro fasi con il nome del documento.

In [4]:
bottle = fetch(f"{SITE}/api/pubblico/bottiglia/{LOT}/1").json()
lot_api = bottle["lotto"]
print("Bottle", bottle["bottiglia"]["numero"], "of", bottle["bottiglia"]["di"])
print(lot_api["codice"], lot_api["cultivar"], lot_api["zona"], lot_api["n_bottiglie"], "bottles, recalled:", lot_api["richiamato"])
for step in bottle["passi"]:
    doc = step["documento"]
    print(step["fase"], step["fase_nome"], doc["filename"], doc["sha256"][:16] + "...")

Bottle 1 of 360
25B001 Frantoio Colline del Chianti 360 bottles, recalled: False
0 Raccolta 25B001-harvest.png 504fefdc128cfead...
1 Molitura 25B001-mill.png 782cc29e0301e6df...
2 Analisi 25B001-report.pdf 42ae129e9d86ae7e...
3 Imbottigliamento 25B001-label.png 02b34b7d4150cb8b...


## 5. Leggi lo stesso lotto direttamente dal contratto

Qui non ci si fida del sito: si interroga la catena Polygon Amoy (chain id 80002). Le funzioni `idDaCodice`, `lotto` e `passi` sono di sola lettura e gratuite. **Cosa dovresti vedere:** chain id 80002, il lotto con 360 bottiglie e quattro fasi con la loro impronta.

In [5]:
# Only the functions we need, in JSON ABI form (the ABI describes how to call a contract).
ABI = [
    {"type": "function", "name": "idDaCodice", "stateMutability": "view",
     "inputs": [{"name": "codice", "type": "string"}], "outputs": [{"name": "", "type": "uint256"}]},
    {"type": "function", "name": "lotto", "stateMutability": "view",
     "inputs": [{"name": "id", "type": "uint256"}],
     "outputs": [{"name": "", "type": "tuple", "components": [
         {"name": "codice", "type": "string"}, {"name": "cultivar", "type": "string"},
         {"name": "zona", "type": "string"}, {"name": "dataRaccolta", "type": "uint64"},
         {"name": "kgOlive", "type": "uint32"}, {"name": "nBottiglie", "type": "uint32"},
         {"name": "richiamato", "type": "bool"}, {"name": "motivoRichiamo", "type": "string"}]}]},
    {"type": "function", "name": "passi", "stateMutability": "view",
     "inputs": [{"name": "id", "type": "uint256"}],
     "outputs": [{"name": "", "type": "tuple[]", "components": [
         {"name": "fase", "type": "uint8"}, {"name": "attore", "type": "address"},
         {"name": "timestamp", "type": "uint64"}, {"name": "descrizione", "type": "string"},
         {"name": "docHash", "type": "bytes32"}]}]},
    {"type": "function", "name": "verificaBottiglia", "stateMutability": "view",
     "inputs": [{"name": "codice", "type": "string"}, {"name": "n", "type": "uint32"}],
     "outputs": [{"name": "esiste", "type": "bool"}, {"name": "richiamato", "type": "bool"}, {"name": "id", "type": "uint256"}]},
]

w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"timeout": 20}))
print("chain id:", w3.eth.chain_id)  # 80002 = Polygon Amoy, a test network
contract = w3.eth.contract(address=Web3.to_checksum_address(CONTRACT), abi=ABI)

token_id = contract.functions.idDaCodice(LOT).call()
lot_chain = contract.functions.lotto(token_id).call()
steps_chain = contract.functions.passi(token_id).call()
print("token id:", token_id)
print("lot on chain:", lot_chain[0], lot_chain[1], lot_chain[2], lot_chain[5], "bottles, recalled:", lot_chain[6])
for s in steps_chain:
    print("phase", s[0], "hash 0x" + s[4].hex()[:16] + "...")

chain id: 80002


token id: 5
lot on chain: 25B001 Frantoio Colline del Chianti 360 bottles, recalled: False
phase 0 hash 0x504fefdc128cfead...
phase 1 hash 0x782cc29e0301e6df...
phase 2 hash 0x42ae129e9d86ae7e...
phase 3 hash 0x02b34b7d4150cb8b...


## 6. Confronta le tre impronte di ogni documento

Per ogni fase: si scarica il documento, si calcola l'impronta, e la si confronta con quella dell'API e con quella scritta sulla catena. **Cosa dovresti vedere:** quattro righe con `True` in tutte le colonne di controllo.

In [6]:
rows = []
for step, on_chain in zip(bottle["passi"], steps_chain):
    doc = step["documento"]
    content = fetch(SITE + doc["url"]).content          # download the document
    computed = sha256_hex(content)                      # fingerprint computed here, on your side
    api_hash = doc["sha256"]                            # fingerprint declared by the site
    chain_hash = on_chain[4].hex()                      # fingerprint written on the chain
    rows.append((step["fase_nome"], doc["filename"], computed == api_hash, computed == chain_hash))

print(f"{'phase':<18}{'file':<24}{'= API':<8}{'= chain'}")
for name, filename, ok_api, ok_chain in rows:
    print(f"{name:<18}{filename:<24}{str(ok_api):<8}{ok_chain}")

phase             file                    = API   = chain
Raccolta          25B001-harvest.png      True    True
Molitura          25B001-mill.png         True    True
Analisi           25B001-report.pdf       True    True
Imbottigliamento  25B001-label.png        True    True


## 7. Il controllo della bottiglia sulla catena

La funzione `verificaBottiglia` risponde a: la bottiglia esiste? il lotto è richiamato? **Cosa dovresti vedere:** valida per le bottiglie 1 e 360, non valida per la 361 e per un lotto sconosciuto.

In [7]:
for lot, n in [(LOT, 1), (LOT, 360), (LOT, 361), ("UNKNOWN", 1)]:
    exists, recalled, token = contract.functions.verificaBottiglia(lot, n).call()
    print(f"lot {lot:<8} bottle {n:<4} exists={exists}  recalled={recalled}  token id={token}")

lot 25B001   bottle 1    exists=True  recalled=False  token id=5


lot 25B001   bottle 360  exists=True  recalled=False  token id=5
lot 25B001   bottle 361  exists=False  recalled=False  token id=5


lot UNKNOWN  bottle 1    exists=False  recalled=False  token id=0


## 8. Prova tu: cambia un solo byte

Si modifica una copia del primo documento in memoria e si ricalcola l'impronta: non coincide più con quella sulla catena. **Cosa dovresti vedere:** `original intact? True` per l'originale e `modified intact? False` per la copia modificata.

In [8]:
original = fetch(SITE + bottle["passi"][0]["documento"]["url"]).content
modified = bytearray(original)
modified[-1] ^= 1                                       # flip one bit of the last byte
hash_on_chain = steps_chain[0][4].hex()

print("original intact?", sha256_hex(original) == hash_on_chain)
print("modified intact?", sha256_hex(bytes(modified)) == hash_on_chain)

original intact? True
modified intact? False


## Cosa hai imparato
- L'impronta SHA-256 identifica un documento: cambia un byte e cambia tutta.
- Sulla catena sta solo l'impronta (32 byte), il documento resta fuori.
- Chiunque può rifare il controllo con letture pubbliche, senza chiavi.

Torna al [tutorial](https://tracciabilita-olio.blockchain.giuseppebosi.com/tutorial/it/) - [codice su GitHub](https://github.com/JoeBosi/pietre-di-vita-demo).